# 🤟 A-Y Sign Language — YOLOv8 Local Pipeline
### (Excludes J and Z — handled separately by LSTM)

## Run order:
| Cell | What it does | Run when |
|------|-------------|----------|
| 1 | Install ultralytics + tools | Once |
| 2 | Config & paths | Every time |
| 3 | Collect images (webcam) | To gather raw images |
| 4 | Label images (LabelImg) | After collecting |
| 5 | Check labeling progress | Anytime |
| 6 | Create train/val split + dataset.yaml | After labeling some |
| 7 | Initial training (small, fast) | After Cell 6 |
| 8 | Auto-label remaining images | After Cell 7 |
| 9 | Final split (manual + auto labels) | After Cell 8 |
| 10 | Final training (better model) | After Cell 9 |
| 11 | Evaluate per-class results | After Cell 10 |
| 12 | Export final model | Last step |

---
# CELL 1 — Install Dependencies
### ▶️ Run ONCE

In [ ]:
import subprocess, sys

packages = ['ultralytics', 'labelImg', 'seaborn', 'pyyaml']
for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

from ultralytics import YOLO
import torch
print('\n✅ Installed!')
print(f'   torch  : {torch.__version__}')
print(f'   CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')

---
# CELL 2 — Config & Setup
### ▶️ Run EVERY TIME

In [ ]:
import os, shutil, random, time
import yaml

# ⬇️ YOUR PROJECT PATH
PROJECT_PATH = r'E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS'

# ── Classes (A-Y excluding J and Z) ───────────────
CLASSES = [c for c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ' if c not in ('J', 'Z')]
print(f'Classes ({len(CLASSES)}): {CLASSES}')

# ── Paths ──────────────────────────────────────────
DATA_PATH    = os.path.join(PROJECT_PATH, 'AY_data')
RAW_PATH     = os.path.join(DATA_PATH, 'raw')          # collected images + labels live here per class
DATASET_PATH = os.path.join(DATA_PATH, 'dataset')      # final YOLO train/val split
MODELS_PATH  = os.path.join(DATA_PATH, 'models')

for c in CLASSES:
    os.makedirs(os.path.join(RAW_PATH, c), exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

# ── Collection settings ───────────────────────────
IMAGES_PER_CLASS = 50   # target images per class

# ── Training settings ─────────────────────────────
INITIAL_MODEL = 'yolov8n.pt'   # nano — fast initial pass for auto-labeling
FINAL_MODEL   = 'yolov8s.pt'   # small — better final accuracy

# ── classes.txt for LabelImg (YOLO format) ────────
classes_txt = os.path.join(DATA_PATH, 'classes.txt')
with open(classes_txt, 'w') as f:
    f.write('\n'.join(CLASSES))

print(f'\n✅ Setup complete!')
print(f'   Raw images path : {RAW_PATH}')
print(f'   Dataset path    : {DATASET_PATH}')
print(f'   Target images   : {IMAGES_PER_CLASS} per class')
print(f'   classes.txt     : {classes_txt}')

---
# CELL 3 — Collect Images (Webcam)
### ▶️ Run to write the collection script, then run it in a SEPARATE terminal
### Controls: SPACE = capture | N = next class | Q = quit
### Tip: vary hand angle, distance, lighting for each shot

In [ ]:
collect_script = f'''
import cv2, os, time

RAW_PATH = r"{RAW_PATH}"
CLASSES  = {CLASSES}
TARGET   = {IMAGES_PER_CLASS}

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(1)

print("Controls: SPACE=capture | N=next class | Q=quit")

class_idx = 0
while class_idx < len(CLASSES):
    current_class = CLASSES[class_idx]
    class_dir = os.path.join(RAW_PATH, current_class)
    existing = len([f for f in os.listdir(class_dir) if f.endswith(".jpg")])
    count = existing

    while count < TARGET:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)
        display = frame.copy()
        cv2.rectangle(display, (0,0), (display.shape[1], 90), (0,0,0), -1)
        cv2.putText(display, f"Class: {{current_class}}  ({{count}}/{{TARGET}})",
                   (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 2)
        cv2.putText(display, "SPACE=capture | N=next | Q=quit",
                   (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
        cv2.imshow("Collecting A-Y Images", display)

        key = cv2.waitKey(1) & 0xFF
        if key == ord(" "):
            fname = f"{{current_class}}_{{int(time.time()*1000)}}.jpg"
            cv2.imwrite(os.path.join(class_dir, fname), frame)
            count += 1
            print(f"  {{current_class}}: {{count}}/{{TARGET}} saved")
        elif key == ord("n"):
            break
        elif key == ord("q"):
            cap.release(); cv2.destroyAllWindows(); exit()

    class_idx += 1

cap.release()
cv2.destroyAllWindows()
print("\\n✅ Collection done!")
'''

script_path = os.path.join(PROJECT_PATH, 'collect_ay.py')
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(collect_script)

print(f'✅ Script saved: {script_path}')
print()
print('Run in a SEPARATE terminal:')
print(f'   cd "{PROJECT_PATH}"')
print( '   ObjEnv\\Scripts\\activate')
print( '   python collect_ay.py')

---
# CELL 4 — Label Images (LabelImg)
### ▶️ Launches LabelImg directly
### ⚠️ IMPORTANT: In LabelImg, click the format button until it says "YOLO"
### Saves .txt label files next to each .jpg automatically
### Shortcut: W = draw box, A/D = prev/next image, Ctrl+S = save

In [ ]:
# ⬇️ Change this to the class you want to label
LABEL_CLASS = 'A'

class_dir = os.path.join(RAW_PATH, LABEL_CLASS)
print(f'Launching LabelImg for class: {LABEL_CLASS}')
print(f'Folder: {class_dir}')
print()
print('⚠️  Click the format toggle button until it shows "YOLO"')
print('   Draw box (W) → select class → Ctrl+S → D for next image')
print()
print('Closing the LabelImg window returns control to this notebook.')

import subprocess
subprocess.Popen([
    sys.executable, '-m', 'labelImg',
    class_dir,
    os.path.join(DATA_PATH, 'classes.txt'),
    class_dir
])

---
# CELL 5 — Check Labeling Progress
### Run anytime to see how much is labeled per class

In [ ]:
print('📊 Labeling Progress')
print('='*50)
print(f'{"Class":<8}{"Images":<10}{"Labeled":<10}{"Unlabeled":<10}')
print('-'*50)

total_images, total_labeled = 0, 0
for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    imgs = [f for f in os.listdir(cdir) if f.endswith('.jpg')]
    labeled = [f for f in imgs if os.path.exists(
        os.path.join(cdir, f.replace('.jpg', '.txt')))]
    total_images  += len(imgs)
    total_labeled += len(labeled)
    print(f'{c:<8}{len(imgs):<10}{len(labeled):<10}{len(imgs)-len(labeled):<10}')

print('-'*50)
print(f'{"TOTAL":<8}{total_images:<10}{total_labeled:<10}{total_images-total_labeled:<10}')
print('='*50)

MIN_LABELED = 15
ready = [c for c in CLASSES if len([
    f for f in os.listdir(os.path.join(RAW_PATH,c)) if f.endswith('.jpg')
    and os.path.exists(os.path.join(RAW_PATH,c,f.replace('.jpg','.txt')))
]) >= MIN_LABELED]

print(f'\nClasses with ≥{MIN_LABELED} labeled images: {len(ready)}/{len(CLASSES)}')
if len(ready) < len(CLASSES):
    missing = [c for c in CLASSES if c not in ready]
    print(f'Still need labeling: {missing}')
else:
    print('✅ Ready for Cell 6!')

---
# CELL 6 — Create Train/Val Split + dataset.yaml
### ▶️ Run after labeling at least ~15 images per class
### Splits 80/20 and writes dataset.yaml for YOLO

In [ ]:
def rebuild_split(source_dirs_per_class, dataset_path, classes, val_ratio=0.2, seed=42):
    """source_dirs_per_class: dict {class_name: [list of (img_path, label_path)]}"""
    random.seed(seed)
    for split in ['train', 'val']:
        os.makedirs(os.path.join(dataset_path, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(dataset_path, 'labels', split), exist_ok=True)

    counts = {'train': 0, 'val': 0}
    for cls, pairs in source_dirs_per_class.items():
        random.shuffle(pairs)
        n_val = max(1, int(len(pairs) * val_ratio))
        val_pairs   = pairs[:n_val]
        train_pairs = pairs[n_val:]

        for split, split_pairs in [('train', train_pairs), ('val', val_pairs)]:
            for img_path, lbl_path in split_pairs:
                img_name = os.path.basename(img_path)
                lbl_name = os.path.basename(lbl_path)
                shutil.copy(img_path, os.path.join(dataset_path, 'images', split, img_name))
                shutil.copy(lbl_path, os.path.join(dataset_path, 'labels', split, lbl_name))
                counts[split] += 1
    return counts


# ── Gather labeled pairs per class ────────────────
labeled_pairs = {}
for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    pairs = []
    for f in os.listdir(cdir):
        if f.endswith('.jpg'):
            lbl = f.replace('.jpg', '.txt')
            lbl_path = os.path.join(cdir, lbl)
            if os.path.exists(lbl_path):
                pairs.append((os.path.join(cdir, f), lbl_path))
    labeled_pairs[c] = pairs
    print(f'  {c}: {len(pairs)} labeled images')

# ── Clean previous dataset and rebuild ────────────
if os.path.exists(DATASET_PATH):
    shutil.rmtree(DATASET_PATH)

counts = rebuild_split(labeled_pairs, DATASET_PATH, CLASSES)
print(f'\n✅ Split done: {counts}')

# ── Write dataset.yaml ─────────────────────────────
dataset_yaml = {
    'path': DATASET_PATH,
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(CLASSES),
    'names': CLASSES
}
yaml_path = os.path.join(DATA_PATH, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f'✅ dataset.yaml written: {yaml_path}')
print(f'\n{yaml.dump(dataset_yaml, default_flow_style=False)}')

---
# CELL 7 — Initial Training (Fast Pass)
### ▶️ Quick yolov8n training — used ONLY to auto-label remaining images
### Takes ~10-20 min on GTX 1650

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

model_initial = YOLO(INITIAL_MODEL)

results = model_initial.train(
    data    = yaml_path,
    epochs  = 50,
    imgsz   = 640,
    batch   = 4,
    device  = 0,
    project = MODELS_PATH,
    name    = 'initial',
    exist_ok= True,
    verbose = True
)

initial_weights = os.path.join(MODELS_PATH, 'initial', 'weights', 'best.pt')
print(f'\n✅ Initial model saved: {initial_weights}')

---
# CELL 8 — Auto-Label Remaining Images
### ▶️ Uses initial model to label unlabeled images
### High-confidence labels accepted automatically
### Low-confidence flagged for manual review

In [ ]:
import json

AUTO_LABEL_CONF = 0.5   # accept auto-labels above this confidence

auto_model = YOLO(initial_weights)

auto_labeled, flagged_review = 0, []

for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    cls_idx = CLASSES.index(c)

    for f in os.listdir(cdir):
        if not f.endswith('.jpg'): continue
        lbl_path = os.path.join(cdir, f.replace('.jpg', '.txt'))
        if os.path.exists(lbl_path):
            continue  # already labeled (manually)

        img_path = os.path.join(cdir, f)
        results  = auto_model(img_path, conf=AUTO_LABEL_CONF, verbose=False)

        if results[0].boxes and len(results[0].boxes) > 0:
            # Use highest-confidence box, force-assign to this folder's class
            box  = results[0].boxes[0]
            xywhn = box.xywhn[0].tolist()  # normalized x_center,y_center,w,h
            conf  = float(box.conf[0])

            with open(lbl_path, 'w') as lf:
                lf.write(f'{cls_idx} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}\n')

            auto_labeled += 1
            if conf < 0.7:
                flagged_review.append({'class': c, 'file': f, 'confidence': round(conf,2)})
        else:
            flagged_review.append({'class': c, 'file': f, 'confidence': 0})

# Save review list
review_path = os.path.join(DATA_PATH, 'manual_review.json')
with open(review_path, 'w') as f:
    json.dump(flagged_review, f, indent=2)

print(f'✅ Auto-labeled: {auto_labeled} images')
print(f'⚠️  Flagged for review (low/no confidence): {len(flagged_review)}')
print(f'   Saved to: {review_path}')
print()
print('Review low-confidence labels by re-running Cell 4 with')
print('LABEL_CLASS set to the flagged classes, and fix/confirm boxes.')

---
# CELL 9 — Final Split (Manual + Auto Labels)
### ▶️ Rebuilds dataset.yaml split using ALL labeled images now

In [ ]:
# ── Gather ALL labeled pairs (manual + auto) ──────
labeled_pairs = {}
for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    pairs = []
    for f in os.listdir(cdir):
        if f.endswith('.jpg'):
            lbl = f.replace('.jpg', '.txt')
            lbl_path = os.path.join(cdir, lbl)
            if os.path.exists(lbl_path):
                pairs.append((os.path.join(cdir, f), lbl_path))
    labeled_pairs[c] = pairs
    print(f'  {c}: {len(pairs)} total labeled images')

if os.path.exists(DATASET_PATH):
    shutil.rmtree(DATASET_PATH)

counts = rebuild_split(labeled_pairs, DATASET_PATH, CLASSES)
print(f'\n✅ Final split: {counts}')

with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)
print(f'✅ dataset.yaml refreshed')

---
# CELL 10 — Final Training
### ▶️ Better model (yolov8s), more epochs, on full dataset
### Takes ~1-2 hours on GTX 1650 — go grab a coffee ☕

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

model_final = YOLO(FINAL_MODEL)

results = model_final.train(
    data     = yaml_path,
    epochs   = 150,
    imgsz    = 640,
    batch    = 4,
    patience = 25,         # early stopping
    device   = 0,
    project  = MODELS_PATH,
    name     = 'final',
    exist_ok = True,
    verbose  = True
)

final_weights = os.path.join(MODELS_PATH, 'final', 'weights', 'best.pt')
print(f'\n✅ Final model saved: {final_weights}')

---
# CELL 11 — Evaluate Per-Class Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from PIL import Image as PILImage
import io

eval_model = YOLO(final_weights)
metrics    = eval_model.val(data=yaml_path)

print('\n📊 PER-CLASS RESULTS')
print('='*50)
print(f'{"Class":<8}{"mAP50":<10}{"Precision":<12}{"Recall":<10}')
print('-'*50)

results_summary = {}
for i, c in enumerate(CLASSES):
    map50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0
    p     = metrics.box.p[i]    if i < len(metrics.box.p)    else 0
    r     = metrics.box.r[i]    if i < len(metrics.box.r)    else 0
    results_summary[c] = map50
    grade = '✅' if map50>=0.8 else '⚠️ ' if map50>=0.5 else '❌'
    print(f'{grade} {c:<6}{map50:<10.2f}{p:<12.2f}{r:<10.2f}')

print('-'*50)
print(f'Overall mAP50: {metrics.box.map50:.2f}')
print('='*50)

# Confusion matrix
cm_path = os.path.join(MODELS_PATH, 'final', 'confusion_matrix.png')
if os.path.exists(cm_path):
    display(PILImage.open(cm_path))

weak = [c for c,v in results_summary.items() if v < 0.7]
if weak:
    print(f'\n⚠️  Weak classes (mAP50 < 0.7): {weak}')
    print('   Consider collecting more images for these in Cell 3')
else:
    print('\n✅ All classes look good!')

---
# CELL 12 — Export Final Model

In [ ]:
import shutil

# Copy best model to project root with clear name
output_path = os.path.join(PROJECT_PATH, 'sign_lang_AY_best.pt')
shutil.copy(final_weights, output_path)
print(f'✅ Model copied to: {output_path}')

# Export to additional formats
export_model = YOLO(final_weights)

print('\nExporting to ONNX...')
export_model.export(format='onnx')

print('Exporting to TFLite (for mobile)...')
export_model.export(format='tflite')

print('\n✅ All exports done!')
print(f'   PyTorch : {output_path}')
print(f'   ONNX/TFLite saved next to: {final_weights}')
print()
print('Use sign_lang_AY_best.pt in your detect_combined.py script')